In [257]:
import sys
!{sys.executable} -m pip install -qq --upgrade openai

In [262]:
!{sys.executable} -m pip uninstall -y typing_extensions

# install latest from GitHub (has TypeIs)
!{sys.executable} -m pip install git+https://github.com/python/typing_extensions.git

Found existing installation: typing_extensions 4.12.2
Uninstalling typing_extensions-4.12.2:
  Successfully uninstalled typing_extensions-4.12.2
  Cloning https://github.com/python/typing_extensions.git to c:\users\vidya\appdata\local\temp\pip-req-build-425q1gjd
  Resolved https://github.com/python/typing_extensions.git to commit 16cc1566a6a4b8e729e8b91e1a4e6a31526823a0
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for typing_extensions: filename=typing_extensions-4.15.0-py3-none-any.whl size=45227 sha256=d5734dd6c5cbb28ad9664e467a05b1197ccce3d41d5768d68a9cdfd87ad32182
  Stored in directory: C:\Users\vidya\AppData\Local\Temp\pip-ephem-wheel-cache-nz8x2une\wheels\f9\8d\16\690d42faa6a8ee38e7b5

  Running command git clone --filter=blob:none --quiet https://github.com/python/typing_extensions.git 'C:\Users\vidya\AppData\Local\Temp\pip-req-build-425q1gjd'


In [261]:
import typing_extensions
print(dir(typing_extensions))

['AbstractSet', 'Annotated', 'Any', 'AnyStr', 'AsyncContextManager', 'AsyncGenerator', 'AsyncIterable', 'AsyncIterator', 'Awaitable', 'BinaryIO', 'Buffer', 'Callable', 'ChainMap', 'ClassVar', 'Collection', 'Concatenate', 'Container', 'ContextManager', 'Coroutine', 'Counter', 'DefaultDict', 'Deque', 'Dict', 'Doc', 'Final', 'ForwardRef', 'FrozenSet', 'Generator', 'Generic', 'GenericMeta', 'Hashable', 'IO', 'IntVar', 'ItemsView', 'Iterable', 'Iterator', 'KT', 'KeysView', 'List', 'Literal', 'LiteralString', 'Mapping', 'MappingView', 'Match', 'MutableMapping', 'MutableSequence', 'MutableSet', 'NamedTuple', 'Never', 'NewType', 'NoReturn', 'NotRequired', 'Optional', 'OrderedDict', 'PEP_560', 'ParamSpec', 'ParamSpecArgs', 'ParamSpecKwargs', 'Pattern', 'Protocol', 'ReadOnly', 'Required', 'Reversible', 'Self', 'Sequence', 'Set', 'Sized', 'SupportsAbs', 'SupportsBytes', 'SupportsComplex', 'SupportsFloat', 'SupportsIndex', 'SupportsInt', 'SupportsRound', 'T', 'TYPE_CHECKING', 'T_co', 'T_contra', '

In [123]:
import pandas as pd
import pandasql as psql
import re
import matplotlib.pyplot as plt

In [148]:
def load_data():
    sales = pd.read_csv("C:\myCODE\AgenticAIReporting\data\sales.csv")
    products = pd.read_csv("C:\myCODE\AgenticAIReporting\data\products.csv")
    stores = pd.read_csv("C:\myCODE\AgenticAIReporting\data\stores.csv")

    table_metadata = """Table: products; Columns: product_id	,product_name ,category	,brand	,price;
                        Table: sales; Columns: sale_id	,product_id	,store_id	,quantity	,sale_date;
                        Table: stores; Columns:store_id	,store_name	,city	,state	,region; """

load_data()

In [95]:
products.sample(3)

,product_id,product_name,category,brand,price
115,116,Product_116,Home,Ikea,1391.49
470,471,Product_471,Sports,Nike,1381.56
43,44,Product_44,Toys,Apple,1500.56


In [115]:
sales.sample(3)

,sale_id,product_id,store_id,quantity,sale_date
35478,35479,835,197,14,2024-10-26
8106,8107,942,159,10,2024-07-27
71785,71786,87,156,12,2024-12-09


In [96]:
stores.sample(3)

,store_id,store_name,city,state,region
116,117,Store_117,Portland,WA,West
151,152,Store_152,Denver,IL,West
103,104,Store_104,San Francisco,CA,West


In [235]:
def load_prompt(Question:str):
        #Question = """show volume of sales, revenue by product category"""

        prompt = """You are a SQL data analyst. You help write sql code and visualize the results in charts.

        Write a SQL query to answer the following question with only the tables/columns provided in metadata: 
        Only return the SQL query starting with the Keyword "SQLQuery": Do not explain.
        Suggest a visualization to display above Result, starting with the Keyword "Chart:".Do not explain. choose one of {bar, pie, line, table}
        Add comment in query field selection to indicate 2 coordinates for the chart. use ("-- x coordinate", "-- y coordinate") Do not rename the query result field names.
        Sample:
        SQLQuery:
        SELECT d.department_name as Department             -- x coordinate (Department)
                , count(e.employee_id) as Employees        -- y coordinate (Employee Count)
        FROM employee e
        JOIN department d on e.department_id=d.department_id

        Chart: bar chart


        Metadata = {table_metadata} 
        Question: {Question} """

        return (prompt)

load_prompt(Question)        

'You are a SQL data analyst. You help write sql code and visualize the results in charts.\n\n        Write a SQL query to answer the following question with only the tables/columns provided in metadata: \n        Only return the SQL query starting with the Keyword "SQLQuery": Do not explain.\n        Suggest a visualization to display above Result, starting with the Keyword "Chart:".Do not explain. choose one of {bar, pie, line, table}\n        Add comment in query field selection to indicate 2 coordinates for the chart. use ("-- x coordinate", "-- y coordinate") Do not rename the query result field names.\n        Sample:\n        SQLQuery:\n        SELECT d.department_name as Department             -- x coordinate (Department)\n                , count(e.employee_id) as Employees        -- y coordinate (Employee Count)\n        FROM employee e\n        JOIN department d on e.department_id=d.department_id\n\n        Chart: bar chart\n\n\n        Metadata = {table_metadata} \n        Qu

In [210]:
def invoke_llm(prompt:str):
    llm_result = """
    SQLQuery:
    SELECT 
        p.category,             -- x coordinate (Product Category)
        SUM(s.quantity) AS total_volume,   -- y coordinate (Sales Volume)
        SUM(s.quantity * p.price) AS total_revenue   
    FROM sales s
    JOIN products p ON s.product_id = p.product_id
    GROUP BY  p.category
    ORDER BY  p.category;

    Chart: Clustered bar chart with Region on X-axis, grouped by Product Category, and two bars per group representing Sales Volume and Revenue.
    """

    return llm_result

invoke_llm("")

'\n    SQLQuery:\n    SELECT \n        p.category,             -- x coordinate (Product Category)\n        SUM(s.quantity) AS total_volume,   -- y coordinate (Sales Volume)\n        SUM(s.quantity * p.price) AS total_revenue   \n    FROM sales s\n    JOIN products p ON s.product_id = p.product_id\n    GROUP BY  p.category\n    ORDER BY  p.category;\n\n    Chart: Clustered bar chart with Region on X-axis, grouped by Product Category, and two bars per group representing Sales Volume and Revenue.\n    '

In [264]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-5-nano",api_key="")

#prompt template
from langchain.prompts import PromptTemplate
template = """You are a SQL data analyst. You help write sql code and visualize the results in charts.

Write a SQL query to answer the following question with only the tables/columns provided in metadata: 
Only return the SQL query starting with the Keyword "SQLQuery": Do not explain.
Suggest a visualization to display above Result, starting with the Keyword "Chart:".Do not explain. choose one of {bar, pie, line, table}
Add comment in query field selection to indicate 2 coordinates for the chart. use ("-- x coordinate", "-- y coordinate") Do not rename the query result field names.
Sample:
SQLQuery:
SELECT d.department_name as Department             -- x coordinate (Department)
        , count(e.employee_id) as Employees        -- y coordinate (Employee Count)
FROM employee e
JOIN department d on e.department_id=d.department_id

Chart: bar chart


Metadata = {table_metadata} 
Question: {question}"""
prompt = PromptTemplate(template=template, input_variables=["metadata","question"])

# Run
table_metadata = """Table: Products; Columns: product_id	,product_name ,category	,brand	,price;
                    Table: Sales; Columns: sale_id	,product_id	,store_id	,quantity	,sale_date;
                    Table: Stores; Columns:store_id	,store_name	,city	,state	,region; """

from langchain.schema import StrOutputParser
chain = prompt | llm | StrOutputParser()

def invoke_llm_openai(question:str):
    response = chain.invoke({"question": question,"metadata":table_metadata})
    return response

ImportError: cannot import name 'TypeIs' from 'typing_extensions' (c:\Users\vidya\anaconda3\Lib\site-packages\typing_extensions.py)

In [231]:
def parse_sql_and_chart(parse_text):
    # Extract SQL query
    sql_query_match = re.search(r"sqlquery:\s*(select[\s\S]+?)\s*chart:", parse_text, re.IGNORECASE)
    sql_query = sql_query_match.group(1).strip() if sql_query_match else None

    # Extract chart suggestion
    chart_match = re.search(r"Chart:\s*(.*)", parse_text, re.IGNORECASE)
    chart_text = chart_match.group(1).strip() 

    # Map common chart types
    if "line" in chart_text:
        chart_type = "line"
    elif "pie" in chart_text:
        chart_type = "pie"
    elif "bar" in chart_text:
        chart_type = "bar"
    else:
        chart_type = "table"

    # Extract x and y coordinates from SQL comments
    # Look for "-- X coordinate (...)" or "-- Y coordinate (...)" in SELECT lines
    x_coord, y_coord = None, None
    select_lines = parse_text.splitlines()
    for line in select_lines:
        if "-- x coordinate" in line:
            if "as" in line:
                x_coord = line.split("as")[1].split()[0].strip().strip(',')  
            else:
                x_coord = line.split(".")[1].split()[0].strip().strip(',')  
        if "-- y coordinate" in line:
            if "as" in line:
                y_coord = line.split("as")[1].split()[0].strip().strip(',')  
            else:
                y_coord = line.split(".")[1].split()[0].strip().strip(',')  

    return {
        "sql_query": sql_query,
        "chart": chart_type,
        "x_coordinate": x_coord,
        "y_coordinate": y_coord
    }

parsed = parse_sql_and_chart(invoke_llm(""))
print("SQLQuery:\n", parsed['sql_query'])
print("Chart:\n", parsed)

SQLQuery:
 SELECT 
        p.category,             -- x coordinate (Product Category)
        SUM(s.quantity) AS total_volume,   -- y coordinate (Sales Volume)
        SUM(s.quantity * p.price) AS total_revenue   
    FROM sales s
    JOIN products p ON s.product_id = p.product_id
    GROUP BY  p.category
    ORDER BY  p.category;
Chart:
 {'sql_query': 'SELECT \n        p.category,             -- x coordinate (Product Category)\n        SUM(s.quantity) AS total_volume,   -- y coordinate (Sales Volume)\n        SUM(s.quantity * p.price) AS total_revenue   \n    FROM sales s\n    JOIN products p ON s.product_id = p.product_id\n    GROUP BY  p.category\n    ORDER BY  p.category;', 'chart': 'bar', 'x_coordinate': 'category', 'y_coordinate': 'quantity)'}


In [157]:
def query_validate(query: str):
    forbidden = ["insert","update","delete","drop","alter","create","truncate","attach"]
    for kw in forbidden:
        if re.search(r"\b"+kw+r"\b" , query.lower()):
            return False, "forbidden keyword found: "+kw
    return True, "Passed"    

query_validate(parsed['sql_query'])   

(True, 'Passed')

In [158]:
def run_sql(query:str):
    result = psql.sqldf(query, globals())  #locals() → pulls from the current local scope (inside a function or block).
    return(result)

run_sql(parsed['sql_query'])

,category,total_volume,total_revenue
0,Accessories,117425,1.159041e+08
1,Books,119661,1.236788e+08
2,Clothing,113495,1.178641e+08
3,Electronics,129875,1.352909e+08
4,Furniture,112217,1.172394e+08
5,Home,147074,1.335571e+08
6,Sports,125144,1.263029e+08
7,Toys,134910,1.365074e+08


In [159]:
def display_bar(x :str, y: str):
    plt.figure(figsize=(8,5))
    plt.bar(result[x], result[y], color='skyblue')
    plt.xlabel(x)
    plt.ylabel(y)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    return

def display_line(x :str, y: str):
    plt.figure(figsize=(8,5))
    plt.plot(result[x], result[y], marker='o', linestyle='-', color='blue')
    plt.xlabel(x)
    plt.ylabel(y)
    plt.xticks(rotation=45)
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    return

def display_pie(x :str, y: str):
    plt.figure(figsize=(7,7))
    plt.pie(result[y], labels=result[x], autopct='%1.1f%%', startangle=140, colors=plt.cm.Paired.colors)
    plt.tight_layout()
    plt.show() 
    return   

def display_grid():
    #plt.figure(figsize=(6,2))
    #plt.axis('off')  # hide axes
    #tbl = plt.table(cellText=result.values, colLabels=result.columns, cellLoc='center', loc='center')
    #tbl.auto_set_font_size(False)
    #tbl.set_fontsize(12)
    #tbl.auto_set_column_width([0,1])
    #plt.show()
    print (result)
    return

def display_chart(chart:str ,x :str, y: str):
    if chart=="bar" :
        display_bar(x , y)
    if chart=="line" :
        display_line(x , y)
    if chart=="pie" :
        display_pie(x , y)
    if chart=="table" :
        display_grid()
    return

In [265]:
load_data()
#print ("data load complete")
#    table_metadata = """Table: products; Columns: product_id	,product_name ,category	,brand	,price;
#                        Table: sales; Columns: sale_id	,product_id	,store_id	,quantity	,sale_date;
#                        Table: stores; Columns:store_id	,store_name	,city	,state	,region; """

Question = "show volume of sales, revenue by product category"
prompt = load_prompt(Question) 
print(f"Prompt:",prompt)

llm_result = invoke_llm(prompt) 
#print(llm_result)

Prompt: You are a SQL data analyst. You help write sql code and visualize the results in charts.

        Write a SQL query to answer the following question with only the tables/columns provided in metadata: 
        Only return the SQL query starting with the Keyword "SQLQuery": Do not explain.
        Suggest a visualization to display above Result, starting with the Keyword "Chart:".Do not explain. choose one of {bar, pie, line, table}
        Add comment in query field selection to indicate 2 coordinates for the chart. use ("-- x coordinate", "-- y coordinate") Do not rename the query result field names.
        Sample:
        SQLQuery:
        SELECT d.department_name as Department             -- x coordinate (Department)
                , count(e.employee_id) as Employees        -- y coordinate (Employee Count)
        FROM employee e
        JOIN department d on e.department_id=d.department_id

        Chart: bar chart


        Metadata = {table_metadata} 
        Question: {Q

In [266]:
llm_result = """SQLQuery:

SELECT s.store_name                                -- x coordinate (Store)
       , SUM(sa.quantity * p.price) AS TotalSales  -- y coordinate (Total Sales)
FROM sales sa
JOIN products p ON sa.product_id = p.product_id
JOIN stores s ON sa.store_id = s.store_id
GROUP BY s.store_name
ORDER BY TotalSales DESC
LIMIT 1;


Chart: table"""

parsed = parse_sql_and_chart(llm_result.lower())
#print(parsed)

valid, msg = query_validate(parsed["sql_query"])
print(valid ,msg)
if valid:
    result = run_sql(parsed['sql_query'])   
    #print (result) 
    display_chart(parsed["chart"],parsed["x_coordinate"] , parsed["y_coordinate"])



True Passed
  store_name   totalsales
0  Store_122  11178661.39
